# Step-Audio-EditX ile anlatım üretimi

Okuma Kütüphanesi için stüdyo kalitesinde anlatım sesi üretir.

**Model:** `stepfun-ai/Step-Audio-EditX`

**Önce:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **T4 GPU**

**Önemli:** Bu model *zero-shot klonlama* ile çalışır — 10-20
saniyelik temiz bir referans kaydı ve o kaydın metni gerekir.
6. hücrede hazır bir örnek kayıt indirilir; kendi sesinizi de
yükleyebilirsiniz.

Hücreleri sırayla çalıştırın. Son hücre `audio-<docid>.zip` indirir;
onu sitedeki `assets/audio/<docid>/` klasörüne açın.


In [ ]:
#@title 1 · GPU ve ortam denetimi
import subprocess, sys, torch
g = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                    '--format=csv,noheader'],
                   capture_output=True, text=True).stdout.strip()
print('GPU      :', g or 'YOK — Çalışma zamanı türünü T4 GPU yapın')
print('torch    :', torch.__version__)
print('python   :', sys.version.split()[0])
print('cuda     :', torch.version.cuda)


In [ ]:
#@title 2 · Metinleri çek ve belgeyi seç
REPO = 'https://github.com/adzetto/reading-library'
import os, json

if not os.path.exists('reading-library'):
    !git clone -q --depth 1 $REPO reading-library
else:
    !cd reading-library && git pull -q

NARR = 'reading-library/narration'
index = json.load(open(f'{NARR}/INDEX.json', encoding='utf-8'))
index.sort(key=lambda r: r['chars'])
print(f"{'belge':22s}{'parça':>7s}{'karakter':>10s}{'~dk ses':>9s}")
print('-'*50)
for r in index:
    print(f"{r['id']:22s}{r['items']:7d}{r['chars']:10d}{r['minutes_est']:9d}")
print('\nEn kısadan başlayın: aesop-fables (~5 dk üretim)')


In [ ]:
#@title 3 · Belge seçimi
DOC_ID = 'aesop-fables'  #@param ['aesop-fables','selfish-giant','tell-tale-heart','alice-rabbit-hole','happy-prince','ugly-duckling','net-feasibility','jadr-2022','doc-b89f','doc-net-tr']
#@markdown Uzun belgeleri bölmek için parça aralığı (0,0 = tümü)
PARCA_BAS = 0  #@param {type:'integer'}
PARCA_SON = 0  #@param {type:'integer'}

doc = json.load(open(f'{NARR}/{DOC_ID}.json', encoding='utf-8'))
items = doc['items']
if PARCA_SON: items = items[PARCA_BAS:PARCA_SON]
elif PARCA_BAS: items = items[PARCA_BAS:]

print(doc['title'][:80])
print(f'{len(items)} / {len(doc["items"])} parça seçildi')
print('ton önerisi :', doc['voice_hint'])
print('en uzun     :', max(x['chars'] for x in items), 'karakter')
print('\nörnek:', items[0]['text'][:180])


## Kurulum

Depo klonlanır, bağımlılıklar kurulur ve iki model indirilir
(tokenizer + EditX). Toplam ~14 GB, ilk çalıştırmada 8-12 dakika.

T4'te bellek sıkışırsa `QUANT = True` yapın (AWQ 4-bit, ~6 GB).


In [ ]:
#@title 4 · Step-Audio-EditX kurulumu  (ilk çalıştırma 8-12 dk)
QUANT = False  #@param {type:'boolean'}

import os, sys
if not os.path.exists('/content/Step-Audio-EditX'):
    !git clone -q https://github.com/stepfun-ai/Step-Audio-EditX.git /content/Step-Audio-EditX

# Colab'ın torch'unu bozmadan yalnızca eksik paketleri kur
PKGS = ('torchaudio transformers accelerate hyperpyyaml onnxruntime-gpu '
        'librosa soundfile diffusers conformer lightning wget openai-whisper')
!pip -q install $PKGS 2>/dev/null | tail -2

from huggingface_hub import snapshot_download
TOK_ID = 'stepfun-ai/Step-Audio-Tokenizer'
MOD_ID = 'stepfun-ai/Step-Audio-EditX-AWQ-4bit' if QUANT else 'stepfun-ai/Step-Audio-EditX'
MODEL_LABEL = MOD_ID

print('tokenizer indiriliyor…')
TOK_PATH = snapshot_download(TOK_ID)
print('model indiriliyor…  (~12 GB)')
MOD_PATH = snapshot_download(MOD_ID)
print('hazır')
print(' tokenizer:', TOK_PATH)
print(' model    :', MOD_PATH)


In [ ]:
#@title 5 · Çıktı klasörü
#@markdown Drive bağlarsanız oturum kopsa da üretilen parçalar durur.
USE_DRIVE = True  #@param {type:'boolean'}
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = f'/content/drive/MyDrive/reading-library-audio/{DOC_ID}'
else:
    OUT = f'/content/out/{DOC_ID}'
os.makedirs(OUT, exist_ok=True)
done = {f.rsplit('.',1)[0] for f in os.listdir(OUT) if f.endswith('.wav')}
print('çıktı klasörü   :', OUT)
print('zaten üretilmiş :', len(done), 'parça')
print('üretilecek      :', len([i for i in items if i['id'] not in done]))


## Referans ses

Zero-shot klonlama için tek bir örnek yeter. Üç seçenek:

1. **Deponun örneği** (varsayılan) — modelin kendi örnek kaydı.
2. **Kendi sesiniz** — sol paneldeki dosya simgesinden `ref.wav`
   yükleyin, `REF_WAV` yolunu ve `REF_TEXT`'e kaydın metnini yazın.
3. **Kendi kaydınızı Colab'da alın** — hücredeki kayıt aracını açın.

Kayıt temiz olmalı: sessiz oda, tek ses, 10-20 saniye, arka plan
müziği yok. Metin birebir doğru olmalı, yoksa taklit bozulur.


In [ ]:
#@title 6 · Referans ses ve model
REF_WAV  = '/content/Step-Audio-EditX/examples/denoise_prompt.wav'  #@param {type:'string'}
REF_TEXT = ''  #@param {type:'string'}
VOICE_LABEL = 'step-audio-editx · klonlanmış anlatım'  #@param {type:'string'}

import sys, os, torchaudio
sys.path.insert(0, '/content/Step-Audio-EditX')

# referans yoksa deponun örneğine düş
if not os.path.exists(REF_WAV):
    import glob
    cand = glob.glob('/content/Step-Audio-EditX/examples/*.wav')
    assert cand, 'referans kayıt bulunamadı — kendi ref.wav dosyanızı yükleyin'
    REF_WAV = cand[0]
    print('varsayılan örnek kullanılıyor:', REF_WAV)

info = torchaudio.info(REF_WAV)
dur = info.num_frames / info.sample_rate
print(f'referans: {REF_WAV}')
print(f'süre    : {dur:.1f} sn  ({info.sample_rate} Hz)')
if dur > 30: print('UYARI: 30 sn üstü referans önerilmez')

# referans metni verilmediyse Whisper ile çıkar
if not REF_TEXT.strip():
    print('\nreferans metni verilmedi — Whisper ile çıkarılıyor…')
    import whisper
    REF_TEXT = whisper.load_model('base').transcribe(REF_WAV)['text'].strip()
    print('çıkarılan metin:', REF_TEXT)

from stepaudio import StepAudioTTS
from tokenizer import StepAudioTokenizer

print('\nmodel yükleniyor…')
tokenizer = StepAudioTokenizer(TOK_PATH)
model = StepAudioTTS(MOD_PATH, tokenizer, model_source='local')
SR = 24000
print('hazır')


## Üretim

Her parça ayrı bir `.wav` olur. Colab kopar ya da hücreyi
durdurursanız **aynı hücreyi yeniden çalıştırın** — üretilmiş
parçaları atlar, kaldığı yerden devam eder.

İlk birkaç parçadan sonra sesi dinleyip beğenmezseniz referans
kaydı değiştirip 6. hücreden itibaren tekrarlayın.


In [ ]:
#@title 7 · Üretim  (kesilirse aynı hücreyi tekrar çalıştırın)
import time, os, torchaudio, traceback

todo = [i for i in items if i['id'] not in done]
print(f'{len(todo)} parça üretilecek\n')
t0, hata = time.time(), 0

for n, it in enumerate(todo, 1):
    try:
        wav, sr = model.clone(prompt_wav_path=REF_WAV,
                              prompt_text=REF_TEXT,
                              target_text=it['text'])
        torchaudio.save(f"{OUT}/{it['id']}.wav", wav.cpu(), sr)
        SR = sr
    except Exception as e:
        hata += 1
        print(f"  ! {it['id']}: {type(e).__name__}: {e}")
        if hata <= 1: traceback.print_exc()
        continue
    if n % 5 == 0 or n == len(todo):
        hz = n / (time.time() - t0)
        kalan = (len(todo) - n) / max(hz, 1e-9) / 60
        print(f'{n}/{len(todo)} · {hz*60:.1f} parça/dk · ~{kalan:.0f} dk kaldı')

print(f'\nbitti · {hata} hata')

# ilk parçayı dinleyin
from IPython.display import Audio, display
ilk = f"{OUT}/{todo[0]['id']}.wav" if todo else f"{OUT}/{items[0]['id']}.wav"
if os.path.exists(ilk):
    print('\nörnek:', items[0]['text'][:120])
    display(Audio(ilk))


In [ ]:
#@title 8 · WAV → OPUS, manifest ve indirme
#@markdown Opus çok daha küçük; konuşma için kalite yeterli.
TO_OPUS = True   #@param {type:'boolean'}
BITRATE = '48k'  #@param {type:'string'}

import os, json, glob, subprocess, wave, contextlib
!apt-get -qq install -y ffmpeg zip > /dev/null 2>&1

fmt  = 'opus' if TO_OPUS else 'wav'
pack = f'/content/pack/{DOC_ID}'
os.makedirs(pack, exist_ok=True)

entries, total, missing = [], 0.0, 0
for it in doc['items']:
    src = f"{OUT}/{it['id']}.wav"
    if not os.path.exists(src):
        missing += 1
        continue
    with contextlib.closing(wave.open(src)) as w:
        dur = round(w.getnframes() / float(w.getframerate()), 2)
    dst = f"{pack}/{it['id']}.{fmt}"
    if TO_OPUS:
        subprocess.run(['ffmpeg','-y','-loglevel','error','-i',src,
                        '-c:a','libopus','-b:a',BITRATE,'-ac','1',dst], check=True)
    else:
        subprocess.run(['cp', src, dst], check=True)
    entries.append({'id': it['id'], 'node': it['node'],
                    'part': it['part'], 'dur': dur})
    total += dur

manifest = {'id': DOC_ID, 'voice': VOICE_LABEL, 'model': MODEL_LABEL,
            'format': fmt, 'sr': SR, 'items': entries}
with open(f'{pack}/manifest.js', 'w', encoding='utf-8') as f:
    f.write('window.AUDIO = window.AUDIO || {};\n')
    f.write(f'window.AUDIO["{DOC_ID}"] = ')
    json.dump(manifest, f, ensure_ascii=False)
    f.write(';\n')

mb = sum(os.path.getsize(p) for p in glob.glob(f'{pack}/*')) / 1048576
print(f'{len(entries)} parça · {total/60:.1f} dakika ses · {mb:.1f} MB')
if missing:
    print(f'UYARI: {missing} parça eksik — 7. hücreyi tekrar çalıştırın')

!cd /content/pack && zip -qr audio-$DOC_ID.zip $DOC_ID
from google.colab import files
files.download(f'/content/pack/audio-{DOC_ID}.zip')


## Siteye koyma

```
1. audio-<docid>.zip dosyasını açın
2. çıkan <docid>/ klasörünü  assets/audio/  altına koyun
3. assets/audio/index.js içine id'yi ekleyin:
     window.AUDIO_INDEX = ['aesop-fables'];
```

Site açıldığında seslendirme panelinde **Stüdyo ses** rozeti belirir.
Paragraf vurgusu ve iki dilli transkript kendiliğinden hizalanır.
